In [1]:
import os
import kagglehub
import numpy as np
import pandas as pd

path = kagglehub.dataset_download("dgomonov/new-york-city-airbnb-open-data")
df = pd.read_csv(os.path.join(path, "AB_NYC_2019.csv"))
print("Shape:", df.shape)
df.head()

Using Colab cache for faster access to the 'new-york-city-airbnb-open-data' dataset.
Shape: (48895, 16)


,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


In [2]:
# R1: dataset size and required column/cardinality conditions
print("Rows:", len(df))
print("Columns:", len(df.columns))

numeric_missing = df.select_dtypes(include=np.number).isna().mean()
cat_distinct = df.select_dtypes(include=["object", "category"]).nunique(dropna=True)

print("Numeric missing percentages:")
print((numeric_missing[numeric_missing > 0] * 100).round(2))
print("Categorical distinct counts:")
print(cat_distinct.sort_values())

assert len(df) >= 1000
assert len(df.columns) >= 8
assert (numeric_missing >= 0.03).any()
assert (cat_distinct < 10).any()
assert (cat_distinct > 30).any()
print("R1 PASS")

Rows: 48895
Columns: 16
Numeric missing percentages:
reviews_per_month    20.56
dtype: float64
Categorical distinct counts:
room_type                  3
neighbourhood_group        5
neighbourhood            221
last_review             1764
host_name              11452
name                   47905
dtype: int64
R1 PASS


In [3]:
# R2: one-row-per-column audit
rows = []
treatment = {
    "id": "drop: identifier",
    "name": "drop raw text + missing indicator",
    "host_id": "drop: identifier",
    "host_name": "drop raw text + missing indicator",
    "neighbourhood_group": "most-frequent impute + one-hot",
    "neighbourhood": "rare grouping + one-hot + handle_unknown=ignore",
    "latitude": "median impute + StandardScaler",
    "longitude": "median impute + StandardScaler",
    "room_type": "most-frequent impute + one-hot",
    "price": "target: log1p",
    "minimum_nights": "median + log1p + RobustScaler",
    "number_of_reviews": "median + log1p + RobustScaler",
    "last_review": "constant date + date-to-days + StandardScaler + indicator",
    "reviews_per_month": "median + log1p + RobustScaler + indicator",
    "calculated_host_listings_count": "median + log1p + RobustScaler",
    "availability_365": "median + MinMaxScaler",
}

for col in df.columns:
    s = df[col]
    if pd.api.types.is_numeric_dtype(s):
        q1, q3 = s.quantile([.25, .75]); iqr = q3 - q1
        detail = f"{s.min():.3f} / {s.max():.3f}"
        outliers = int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum())
    else:
        detail = "; ".join(f"{k}: {v}" for k, v in s.value_counts(dropna=True).head(3).items())
        outliers = "N/A"
    rows.append([col, str(s.dtype), round(s.isna().mean()*100, 2), s.nunique(dropna=True), detail, outliers, treatment[col]])

audit = pd.DataFrame(rows, columns=["column", "dtype", "% missing", "distinct", "min / max or top 3", "outliers", "proposed treatment"])
audit

,column,dtype,% missing,distinct,min / max or top 3,outliers,proposed treatment
0,id,int64,0.00,48895,2539.000 / 36487245.000,0,drop: identifier
1,name,object,0.03,47905,Hillside Hotel: 18; Home away from home: 17; N...,N/A,drop raw text + missing indicator
2,host_id,int64,0.00,37457,2438.000 / 274321313.000,1526,drop: identifier
3,host_name,object,0.04,11452,Michael: 417; David: 403; Sonder (NYC): 327,N/A,drop raw text + missing indicator
4,neighbourhood_group,object,0.00,5,Manhattan: 21661; Brooklyn: 20104; Queens: 5666,N/A,most-frequent impute + one-hot
5,neighbourhood,object,0.00,221,Williamsburg: 3920; Bedford-Stuyvesant: 3714; ...,N/A,rare grouping + one-hot + handle_unknown=ignore
6,latitude,float64,0.00,19048,40.500 / 40.913,425,median impute + StandardScaler
7,longitude,float64,0.00,14718,-74.244 / -73.713,2833,median impute + StandardScaler
8,room_type,object,0.00,3,Entire home/apt: 25409; Private room: 22326; S...,N/A,most-frequent impute + one-hot
9,price,int64,0.00,674,0.000 / 10000.000,2972,target: log1p


In [4]:
# R3: missing columns, mechanism evidence, indicators, and before/after summaries
missing_columns = [c for c in df.columns if df[c].isna().any()]
print("Missing columns:", missing_columns)

for col in missing_columns:
    df[col + "_missing"] = df[col].isna().astype(int)

for col in ["reviews_per_month", "last_review"]:
    m = df[col].isna()
    print("\n", col)
    print("Missing:", m.sum())
    print("Mean number_of_reviews, missing:", df.loc[m, "number_of_reviews"].mean())
    print("Mean number_of_reviews, present:", df.loc[~m, "number_of_reviews"].mean())
    print("Zero-review fraction, missing:", (df.loc[m, "number_of_reviews"] == 0).mean())
    print("Zero-review fraction, present:", (df.loc[~m, "number_of_reviews"] == 0).mean())

for col in ["name", "host_name"]:
    print("\n", col, "missing % by room_type")
    print((df.groupby("room_type")[col].apply(lambda x: x.isna().mean()) * 100).round(3))
    print(col, "missing % by neighbourhood_group")
    print((df.groupby("neighbourhood_group")[col].apply(lambda x: x.isna().mean()) * 100).round(3))

Missing columns: ['name', 'host_name', 'last_review', 'reviews_per_month']

 reviews_per_month
Missing: 10052
Mean number_of_reviews, missing: 0.0
Mean number_of_reviews, present: 29.297556831346704
Zero-review fraction, missing: 1.0
Zero-review fraction, present: 0.0

 last_review
Missing: 10052
Mean number_of_reviews, missing: 0.0
Mean number_of_reviews, present: 29.297556831346704
Zero-review fraction, missing: 1.0
Zero-review fraction, present: 0.0

 name missing % by room_type
room_type
Entire home/apt    0.028
Private room       0.036
Shared room        0.086
Name: name, dtype: float64
name missing % by neighbourhood_group
neighbourhood_group
Bronx            0.092
Brooklyn         0.030
Manhattan        0.042
Queens           0.000
Staten Island    0.000
Name: name, dtype: float64

 host_name missing % by room_type
room_type
Entire home/apt    0.035
Private room       0.054
Shared room        0.000
Name: host_name, dtype: float64
host_name missing % by neighbourhood_group
neighb

In [5]:
# R3: before/after mean, median, std
before = df["reviews_per_month"]
after = before.fillna(before.median())
print("reviews_per_month")
print(pd.DataFrame({"stage":["before","after"], "mean":[before.mean(),after.mean()], "median":[before.median(),after.median()], "std":[before.std(),after.std()]}).round(4))

days = (pd.to_datetime(df["last_review"]) - pd.to_datetime(df["last_review"]).min()).dt.days
filled_days = days.fillna(0)
print("\nlast_review, days since first observed date")
print(pd.DataFrame({"stage":["before","after"], "mean":[days.mean(),filled_days.mean()], "median":[days.median(),filled_days.median()], "std":[days.std(),filled_days.std()]}).round(2))

print("\nname / host_name: nominal text, so mean/median/std are N/A; raw text is dropped and the missing indicator is retained.")

reviews_per_month
    stage    mean  median     std
0  before  1.3732    0.72  1.6804
1   after  1.2389    0.72  1.5209

last_review, days since first observed date
    stage     mean  median      std
0  before  2747.07  2974.0   413.92
1   after  2182.32  2838.0  1169.87

name / host_name: nominal text, so mean/median/std are N/A; raw text is dropped and the missing indicator is retained.


In [6]:
# R4: skew/tail evidence and scaler choice
numeric_columns = [c for c in df.select_dtypes(include=np.number).columns if not c.endswith("_missing")]
rows=[]
for col in numeric_columns:
    s=df[col].dropna()
    if col in ["id", "host_id"]:
        choice="drop: identifier"; skew=np.nan; out=np.nan
    elif col == "price":
        choice="target: log1p"; skew=s.skew(); out=np.nan
    else:
        q1,q3=s.quantile([.25,.75]); iqr=q3-q1
        out=((s < q1-1.5*iqr) | (s > q3+1.5*iqr)).mean()*100
        skew=s.skew()
        choice="log1p + RobustScaler" if skew > 1 else ("MinMaxScaler" if col=="availability_365" else "StandardScaler")
    rows.append([col, round(skew,3) if pd.notna(skew) else np.nan, round(out,2) if pd.notna(out) else np.nan, choice])

pd.DataFrame(rows, columns=["column","skew","IQR-outlier %","scaler / transform"])

,column,skew,IQR-outlier %,scaler / transform
0,id,NaN,NaN,drop: identifier
1,host_id,NaN,NaN,drop: identifier
2,latitude,0.237,0.87,StandardScaler
3,longitude,1.284,5.79,log1p + RobustScaler
4,price,19.119,NaN,target: log1p
5,minimum_nights,21.827,13.51,log1p + RobustScaler
6,number_of_reviews,3.691,12.31,log1p + RobustScaler
7,reviews_per_month,3.130,4.62,log1p + RobustScaler
8,calculated_host_listings_count,7.933,14.48,log1p + RobustScaler
9,availability_365,0.763,0.00,MinMaxScaler


In [7]:
# R4: log1p reduces the right-tail skew in the selected numeric columns
for col in ["minimum_nights", "number_of_reviews", "reviews_per_month", "calculated_host_listings_count"]:
    transformed = np.log1p(df[col].fillna(df[col].median()))
    print(col, "original skew:", round(df[col].skew(),3), "log1p skew:", round(transformed.skew(),3))

minimum_nights original skew: 21.827 log1p skew: 1.499
number_of_reviews original skew: 3.691 log1p skew: 0.363
reviews_per_month original skew: 3.13 log1p skew: 0.957
calculated_host_listings_count original skew: 7.933 log1p skew: 3.28


In [8]:
# R5: categorical cardinality and encoding choice
for col in ["neighbourhood_group", "neighbourhood", "room_type"]:
    print(col, "distinct =", df[col].nunique(dropna=True))
print("Low-cardinality categoricals: one-hot encoding.")
print("neighbourhood: min_frequency=20 groups rare levels before one-hot encoding.")
print("No ordered categorical variable is used, so ordinal encoding is not needed.")

neighbourhood_group distinct = 5
neighbourhood distinct = 221
room_type distinct = 3
Low-cardinality categoricals: one-hot encoding.
neighbourhood: min_frequency=20 groups rare levels before one-hot encoding.
No ordered categorical variable is used, so ordinal encoding is not needed.


In [9]:
# R6: required single pipeline structure
from pipeline import build_pipeline
model = build_pipeline()
print(model)
print("Pipeline steps:", list(model.named_steps))

Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('standard',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(keep_empty_features=True,
                                                                                 strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['latitude', 'longitude']),
                                                 ('log_robust',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(keep_empty_features=True,
                                                                                 strategy='median')),
                                 